In [ ]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
import torch

# ===== 1. Mount Google Drive =====
from google.colab import drive
drive.mount('/content/drive')

# ===== 2. Model Path =====
model_path = "/content/drive/MyDrive/mBART_new/OE2ME_models/best_model"

# ===== 3. Load Tokenizer =====
tokenizer = MBart50TokenizerFast.from_pretrained(model_path)

# ===== 4. Load Model =====
model = MBartForConditionalGeneration.from_pretrained(model_path)

# ===== 5. Move to GPU (H100) =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# ===== 6. Set source & target language =====
tokenizer.src_lang = "en_XX"   # (Old English mapped to English)
tgt_lang = "en_XX"

print("Model Loaded Successfully ✅")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

Model Loaded Successfully ✅


In [ ]:
import pandas as pd
import torch
from tqdm.auto import tqdm

# ===== 1. Dataset Path =====
input_file = "/content/D_ChronicleA.xlsx"
output_file = "/content/drive/MyDrive/mBART_new/OE2ME_models/best_model/chronicleA_translated.xlsx"

# ===== 2. Load Dataset =====
df = pd.read_excel(input_file)

# column name စစ်
print(df.columns)

# ===== 3. Keep sentence as string =====
df["sentence"] = df["sentence"].astype(str)

# ===== 4. Translation Function =====
def translate_batch(sentences, model, tokenizer, device, max_length=128):
    inputs = tokenizer(
        sentences,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length
    ).to(device)

    with torch.no_grad():
        generated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.lang_code_to_id["en_XX"],
            max_length=max_length,
            num_beams=5
        )

    outputs = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )
    return outputs

# ===== 5. Batch Translation =====
batch_size = 16   # H100 မှာ 16 နဲ့စလိုက်၊ memory အဆင်ပြေရင် 32 တင်လို့ရ
translated_sentences = []

sentences = df["sentence"].tolist()

for i in tqdm(range(0, len(sentences), batch_size)):
    batch = sentences[i:i+batch_size]
    batch_translations = translate_batch(batch, model, tokenizer, device)
    translated_sentences.extend(batch_translations)

# ===== 6. Add translated column =====
df["translated_sentence"] = translated_sentences

# ===== 7. Save Result =====
df.to_excel(output_file, index=False)

print("Saved to:", output_file)
print(df[["sentence", "translated_sentence"]].head())

Index(['sentence', 'gold_names'], dtype='object')


  0%|          | 0/15 [00:00<?, ?it/s]

Saved to: /content/drive/MyDrive/mBART_new/OE2ME_models/best_model/chronicleA_translated.xlsx
                                            sentence  \
0  þa Cerdic 7 Cynric his sunu cuom up æt Cerdice...   
1  7 þa he gefor, þa feng his sunu Cynric to þam ...   
2  Þa he gefor, þa feng Ceol to þam rice 7 heold ...   
3   Þa feng Cynegils Ceolwulfes broþur sunu to ri...   
4  7 se Cenwalh wæs Cynegilses sunu; 7 þa heold S...   

                                 translated_sentence  
0  Then Hrothgar’s son, Hrothgar the Chaldean, we...  
1  And when he was old, he took his son, Cynric, ...  
2  When he was ready, Ceol took him to the realm,...  
3  Then the brother of Kingel’s, Ceolwolf, took h...  
4  The Ching-wolf was the son of Kingel; and Seax...  


In [ ]:
import pandas as pd
from transformers import pipeline
from tqdm.auto import tqdm

# ===== 1. File Path =====
input_file = "/content/drive/MyDrive/mBART_new/OE2ME_models/best_model/chronicleA_translated.xlsx"
output_file = "/content/drive/MyDrive/mBART_new/OE2ME_models/best_model/chronicleA_translated_with_extracted_names.xlsx"

# ===== 2. Load Dataset =====
df = pd.read_excel(input_file)

# translated_sentence ကို string အဖြစ်သေချာလုပ်
df["translated_sentence"] = df["translated_sentence"].fillna("").astype(str)

# ===== 3. Load RoBERTa-based NER Pipeline =====
# Jean-Baptiste/roberta-large-ner-english is a strong pretrained English NER model
ner_pipe = pipeline(
    "ner",
    model="Jean-Baptiste/roberta-large-ner-english",
    tokenizer="Jean-Baptiste/roberta-large-ner-english",
    aggregation_strategy="simple",
    device=0   # Colab GPU
)

# ===== 4. PERSON extraction function =====
def extract_person_names(text):
    try:
        if not text.strip():
            return ""

        entities = ner_pipe(text)
        person_names = []

        for ent in entities:
            if ent["entity_group"] == "PER":
                name = ent["word"].strip()

                # duplicate မထည့်အောင်
                if name and name not in person_names:
                    person_names.append(name)

        return ", ".join(person_names)

    except Exception as e:
        print("Error:", e)
        return ""

# ===== 5. Run extraction sentence by sentence =====
extracted_names = []

for text in tqdm(df["translated_sentence"], total=len(df)):
    names = extract_person_names(text)
    extracted_names.append(names)

# ===== 6. Add new column =====
df["extracted_names"] = extracted_names

# ===== 7. Save Result =====
df.to_excel(output_file, index=False)

print("Saved to:", output_file)
print(df[["translated_sentence", "extracted_names"]].head())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/849 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

RobertaForTokenClassification LOAD REPORT from: Jean-Baptiste/roberta-large-ner-english
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

  0%|          | 0/236 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Saved to: /content/drive/MyDrive/mBART_new/OE2ME_models/best_model/chronicleA_translated_with_extracted_names.xlsx
                                 translated_sentence  \
0  Then Hrothgar’s son, Hrothgar the Chaldean, we...   
1  And when he was old, he took his son, Cynric, ...   
2  When he was ready, Ceol took him to the realm,...   
3  Then the brother of Kingel’s, Ceolwolf, took h...   
4  The Ching-wolf was the son of Kingel; and Seax...   

                extracted_names  
0                      Hrothgar  
1                        Cynric  
2                Ceol, Ceolwolf  
3    Kingel, Ceolwolf, Shenwolf  
4  Ching-wolf, Kingel, Seaxburg  


In [ ]:
!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 139.0 MB/s eta 0:00:00


In [ ]:
!pip install rapidfuzz openpyxl -q

In [ ]:
import pandas as pd
import re
import unicodedata
from rapidfuzz import fuzz

# ===== 1. File Paths =====
input_file = "/content/drive/MyDrive/mBART_new/OE2ME_models/best_model/chronicleA_translated_with_extracted_names.xlsx"
detailed_output_file = "/content/drive/MyDrive/mBART_new/OE2ME_models/best_model/chronicleA_evaluated_v2.xlsx"
summary_output_file = "/content/drive/MyDrive/mBART_new/OE2ME_models/best_model/chronicleA_metrics_summary_v2.xlsx"

# ===== 2. Load Dataset =====
df = pd.read_excel(input_file)

df["gold_names"] = df["gold_names"].fillna("").astype(str)
df["extracted_names"] = df["extracted_names"].fillna("").astype(str)

# ===== 3. Normalization Helpers =====
def strip_diacritics(text):
    text = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in text if not unicodedata.combining(ch))

def reduce_double_letters(text):
    return re.sub(r'(.)\1+', r'\1', text)

def strip_inflection(name):
    """
    Old English inflection endings ကို အကြမ်းဖျင်းဖြုတ်
    too aggressive မဖြစ်အောင် length check ထည့်ထား
    """
    endings = ["es", "as", "um", "an", "on", "e", "a", "s"]
    changed = True

    while changed:
        changed = False
        for end in endings:
            if name.endswith(end) and len(name) - len(end) >= 4:
                name = name[:-len(end)]
                changed = True
                break
    return name

# ===== 4. Main Normalization =====
def normalize_name(name):
    name = str(name).strip().lower()

    replacements = {
        "æ": "ae",
        "ǽ": "ae",
        "œ": "oe",
        "þ": "th",
        "ð": "d",
        "ł": "l",
        "ß": "ss",
        "ȝ": "g",
    }
    for old, new in replacements.items():
        name = name.replace(old, new)

    name = strip_diacritics(name)

    # punctuation / symbols ဖယ်
    name = re.sub(r"[^a-z0-9\s-]", "", name)

    # hyphen / spaces normalize
    name = re.sub(r"[-]+", " ", name)
    name = re.sub(r"\s+", " ", name).strip()

    # common spelling normalization
    name = name.replace("wulf", "wolf")
    name = name.replace("ae", "e")
    name = name.replace("heard", "hard")

    # double letters reduce
    name = reduce_double_letters(name)

    # spaces ဖယ်
    name = name.replace(" ", "")

    # inflection endings ဖြုတ်
    name = strip_inflection(name)

    return name

# ===== 5. Split names from cell =====
def split_names(text):
    if not text.strip():
        return []

    parts = [x.strip() for x in text.split(",")]
    parts = [x for x in parts if x]

    cleaned = []
    seen = set()

    for p in parts:
        norm = normalize_name(p)
        if norm and norm not in seen:
            seen.add(norm)
            cleaned.append(p.strip())

    return cleaned

# ===== 6. Similarity Function =====
def similarity_score(a, b):
    a_norm = normalize_name(a)
    b_norm = normalize_name(b)

    score1 = fuzz.ratio(a_norm, b_norm)
    score2 = fuzz.partial_ratio(a_norm, b_norm)
    score3 = fuzz.token_sort_ratio(a_norm, b_norm)

    return max(score1, score2, score3)

# ===== 7. One-to-one Matching =====
def match_names(gold_list, pred_list, fuzzy_threshold=80):
    gold_norm = [normalize_name(x) for x in gold_list]
    pred_norm = [normalize_name(x) for x in pred_list]

    matched_gold = set()
    matched_pred = set()
    matched_pairs = []

    # Pass 1: exact match after normalization
    for i, g in enumerate(gold_norm):
        if not g:
            continue
        for j, p in enumerate(pred_norm):
            if i in matched_gold or j in matched_pred:
                continue
            if g == p:
                matched_gold.add(i)
                matched_pred.add(j)
                matched_pairs.append((gold_list[i], pred_list[j], "exact", 100))
                break

    # Pass 2: fuzzy match for remaining
    for i, g in enumerate(gold_norm):
        if i in matched_gold or not g:
            continue

        best_j = None
        best_score = -1

        for j, p in enumerate(pred_norm):
            if j in matched_pred or not p:
                continue

            score = similarity_score(gold_list[i], pred_list[j])

            if score > best_score:
                best_score = score
                best_j = j

        if best_j is not None and best_score >= fuzzy_threshold:
            matched_gold.add(i)
            matched_pred.add(best_j)
            matched_pairs.append((gold_list[i], pred_list[best_j], "fuzzy", best_score))

    tp = len(matched_pairs)
    fp = len(pred_list) - len(matched_pred)
    fn = len(gold_list) - len(matched_gold)

    unmatched_gold = [gold_list[i] for i in range(len(gold_list)) if i not in matched_gold]
    unmatched_pred = [pred_list[j] for j in range(len(pred_list)) if j not in matched_pred]

    return tp, fp, fn, matched_pairs, unmatched_gold, unmatched_pred

# ===== 8. Metric Function =====
def compute_scores(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

# ===== 9. Evaluate Row by Row =====
FUZZY_THRESHOLD = 80

all_tp, all_fp, all_fn = 0, 0, 0

row_tp = []
row_fp = []
row_fn = []
row_precision = []
row_recall = []
row_f1 = []
row_matched = []
row_unmatched_gold = []
row_unmatched_pred = []

for _, row in df.iterrows():
    gold_list = split_names(row["gold_names"])
    pred_list = split_names(row["extracted_names"])

    tp, fp, fn, matched_pairs, unmatched_gold, unmatched_pred = match_names(
        gold_list, pred_list, fuzzy_threshold=FUZZY_THRESHOLD
    )

    precision, recall, f1 = compute_scores(tp, fp, fn)

    row_tp.append(tp)
    row_fp.append(fp)
    row_fn.append(fn)
    row_precision.append(precision)
    row_recall.append(recall)
    row_f1.append(f1)

    row_matched.append("; ".join([f"{g} <-> {p} ({m}:{s})" for g, p, m, s in matched_pairs]))
    row_unmatched_gold.append(", ".join(unmatched_gold))
    row_unmatched_pred.append(", ".join(unmatched_pred))

    all_tp += tp
    all_fp += fp
    all_fn += fn

# ===== 10. Add Results to DataFrame =====
df["tp"] = row_tp
df["fp"] = row_fp
df["fn"] = row_fn
df["precision"] = row_precision
df["recall"] = row_recall
df["f1_score"] = row_f1
df["matched_pairs"] = row_matched
df["unmatched_gold"] = row_unmatched_gold
df["unmatched_predicted"] = row_unmatched_pred

# ===== 11. Micro-average Metrics =====
micro_precision, micro_recall, micro_f1 = compute_scores(all_tp, all_fp, all_fn)

summary_df = pd.DataFrame([{
    "fuzzy_threshold": FUZZY_THRESHOLD,
    "total_tp": all_tp,
    "total_fp": all_fp,
    "total_fn": all_fn,
    "micro_precision": micro_precision,
    "micro_recall": micro_recall,
    "micro_f1": micro_f1
}])

# ===== 12. Save Files =====
df.to_excel(detailed_output_file, index=False)
summary_df.to_excel(summary_output_file, index=False)

# ===== 13. Print Summary =====
print("Detailed file saved to:", detailed_output_file)
print("Summary file saved to:", summary_output_file)
print("\n=== MICRO SCORES ===")
print("TP:", all_tp)
print("FP:", all_fp)
print("FN:", all_fn)
print("Micro Precision:", round(micro_precision, 4))
print("Micro Recall:", round(micro_recall, 4))
print("Micro F1:", round(micro_f1, 4))

Detailed file saved to: /content/drive/MyDrive/mBART_new/OE2ME_models/best_model/chronicleA_evaluated_v2.xlsx
Summary file saved to: /content/drive/MyDrive/mBART_new/OE2ME_models/best_model/chronicleA_metrics_summary_v2.xlsx

=== MICRO SCORES ===
TP: 265
FP: 120
FN: 183
Micro Precision: 0.6883
Micro Recall: 0.5915
Micro F1: 0.6363
